# Corrected and organized bivariate covariance MLE

This replaces the duplicated workflows in **MLE.ipynb** with one implementation
for **G_even, G_odd, G_mixed, M_even, M_odd and M_mixed**.

**Start here:** edit the configuration in **Section 7**, set the CSV path, then
run the notebook from the top. No helper `.py` files are required. Existing data
are never overwritten. Fit results are checkpointed to a new directory.

Changes from the old notebook:

- Correct Gaussian exponent, coefficient and missing $h^p$ factor.
- Correct Matérn normalization and enforce the smoothness/validity restrictions.
- Keep the even and odd Matérn smoothness parameters distinct by name.
- Replace eigenvalue clipping with parameters satisfying spectral validity.
- Cache distances and angular terms; evaluate radial kernels at unique distances.
- Use one Cholesky factorization and a batched solve per likelihood evaluation.
- Use reproducible multistarts, preserve failed/unconverged diagnostics, and save
  all fits without filtering inconvenient estimates.

The original notebook's saved estimates are not carried forward. Corrected fits
must be rerun. This notebook does **not** automatically rerun the manuscript's
Monte Carlo parameter-recovery experiments.

## 1. Environment

Compatible with Python 3.8 and SciPy 1.10 or newer; tested in the original
`deeponet` environment. Required packages are NumPy, SciPy and pandas.
Matplotlib is used for the optional comparison plot. `threadpoolctl` is optional
and limits BLAS threads to avoid overhead on small dense matrices.

If needed, install in the selected notebook kernel with:
`%pip install numpy scipy pandas matplotlib threadpoolctl`.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from functools import lru_cache
from contextlib import nullcontext
import math
import time
import warnings
import numpy as np
import pandas as pd
from scipy.linalg import cholesky, solve_triangular
from scipy.optimize import minimize
from scipy.special import eval_chebyt, eval_gegenbauer, gammaln, kve
from scipy.stats import qmc



import scipy
import sys
print("Python:", sys.version.split()[0], " NumPy:", np.__version__, " SciPy:", scipy.__version__, " pandas:", pd.__version__)

## 2. Fourier formulas and spectral validity

The lag convention is $C_{jk}(h)=\mathrm{Cov}\{Z_j(s),Z_k(s+h)\}$,
$C(h)=\int e^{iu\cdot h}f(u)\,du$, and $f_{12}=c_{12}-iq_{12}$.
In two dimensions $A_{p,2}(t)=T_p(t)$, and $p$ is a positive odd integer.

\[
G_{\rm odd}(h)=(-1)^{(p-1)/2}(2/a^2)^p\|h\|^p
 T_p(\widehat h\cdot u_0)e^{-\|h\|^2/a^2}.
\]
\[
M_{\rm odd}(h)=(-1)^{(p-1)/2}\frac{2^{1-\nu}}{\Gamma(\nu)}
 (a\|h\|)^\nu K_{p-\nu}(a\|h\|)T_p(\widehat h\cdot u_0),
\qquad\nu>p/2.
\]
Odd kernels are zero at exactly zero lag; even kernels have unit sill.

Let $M_e,M_o$ be upper bounds on the **squared** cross-to-marginal spectral
ratio with unit amplitude. Gaussian pure-component maxima are explicit.
Matérn maxima are evaluated at all stationary polynomial roots and analytic
endpoints, with a conservative analytic fallback for ill-conditioned roots.
These are continuous-frequency checks, not finite-grid certificates.

The mixed model uses the **sufficient**, potentially conservative bound
$M=\cos^2\theta M_e+\sin^2\theta M_o$. We fit
$\rho=\texttt{rho_fraction}/\sqrt{M}$, with $|\texttt{rho_fraction}|<1$.
The actual odd amplitude $\rho$ is not restricted to $[-1,1]$.
The mixed collocated correlation is $\rho\cos\theta$.

For Matérn, $\nu_o\ge(\nu_{11}+\nu_{22}+p)/2$ ensures a finite spectral
ratio. The exact odd spectrum is checked with its original normalization.
The comparison quantity
$\rho_{\rm eff}=\rho\,\nu_o/(\nu_o-p/2)$ in dimension two is **reported only**;
it is not multiplied into the actual odd covariance a second time.

The optimizer searches a sufficient mixed-model subset. Failure of this bound
does not prove that a model is invalid under its exact determinant. Floating
point checks have conservative fallbacks and a small upward margin; they are
not interval-arithmetic proofs.

In [ ]:
"""Continuous-frequency spectral bounds for the corrected covariance models.

The pure-component calculation maximizes the squared cross/marginal spectral
ratio over every frequency. Matérn stationary points are roots of a polynomial
of degree at most three; this is not a frequency-grid test. Analytic endpoint
limits are included. Equal-scale Matérn and Gaussian cases have closed forms.

Floating-point polynomial solutions are checked for residuals, reconstruction,
and clustered roots. Unreliable roots trigger an analytic conservative upper
bound instead. The returned finite bounds include a relative 1e-10 upward
margin; they are numerical spectral bounds, not interval-arithmetic proofs.
The mixed bound is sufficient and may restrict the full admissible family.
"""

import math
from decimal import Decimal, localcontext

import numpy as np
from scipy.special import gammaln


_LOG_BOUND_MARGIN = math.log1p(1.0e-10)


def _positive_number(value, name):
    value = float(value)
    if not math.isfinite(value) or value <= 0.0:
        raise ValueError("{} must be positive and finite".format(name))
    return value


def _log_b(nu, d):
    # Gamma(nu+1)/Gamma(nu)=nu avoids cancellation in the planar case.
    return math.log(nu) if d == 2 else float(gammaln(nu + d / 2.0) - gammaln(nu))


def _log_beta_max(k, delta):
    """log max_{0<=t<=1} t**k*(1-t)**delta, including boundaries."""
    if k == 0 or delta == 0.0:
        return 0.0
    total = k + delta
    return k * math.log(k / total) + delta * math.log(delta / total)


def _matern_log_sup(a11, a22, ac, nu11, nu22, nuc, k, d):
    # A negative exponent deficit makes the spectral ratio unbounded.
    delta = math.fsum([2.0 * nuc, -nu11, -nu22, -float(k)])
    if delta < 0.0:
        return math.inf
    log_r1 = 2.0 * (math.log(a11) - math.log(ac))
    log_r2 = 2.0 * (math.log(a22) - math.log(ac))
    alpha, beta = nu11 + d / 2.0, nu22 + d / 2.0
    log_constant = (2.0 * _log_b(nuc, d) - _log_b(nu11, d)
                    - _log_b(nu22, d) - nu11 * log_r1 - nu22 * log_r2)
    log_beta_max = _log_beta_max(k, delta)

    # With t=x/(ac**2+x), each linear term lies between ri and one.
    # Maximizing factors separately therefore supplies an analytic fallback.
    fallback = (log_constant + log_beta_max
                + alpha * max(log_r1, 0.0) + beta * max(log_r2, 0.0))
    if a11 == ac and a22 == ac:
        return log_constant + log_beta_max + _LOG_BOUND_MARGIN

    # Dimensionless y=x/ac**2 prevents units from conditioning the polynomial.
    # Extreme scale ratios use the overflow-safe logarithmic fallback.
    if max(abs(log_r1), abs(log_r2)) > 250.0:
        return fallback + _LOG_BOUND_MARGIN
    r1, r2 = math.exp(log_r1), math.exp(log_r2)
    gamma = 2.0 * nuc + d
    if k:
        coefficients = np.array([
            float(k) * r1 * r2,
            k * (r1 * r2 + r1 + r2) + alpha * r2 + beta * r1 - gamma * r1 * r2,
            k * (r1 + r2 + 1.0) + alpha * (r2 + 1.0)
            + beta * (r1 + 1.0) - gamma * (r1 + r2),
            -delta,
        ], dtype=float)
    else:
        coefficients = np.array([
            alpha * r2 + beta * r1 - gamma * r1 * r2,
            alpha * (r2 + 1.0) + beta * (r1 + 1.0) - gamma * (r1 + r2),
            -delta,
        ], dtype=float)
    # Drop only exact zero leading coefficients, never a small tail deficit.
    while len(coefficients) > 1 and coefficients[-1] == 0.0:
        coefficients = coefficients[:-1]
    coefficient_scale = np.max(np.abs(coefficients))
    if not np.all(np.isfinite(coefficients)):
        return fallback + _LOG_BOUND_MARGIN

    # Evaluate in log(y); logaddexp handles arbitrarily large positive roots.
    def log_ratio(log_y):
        log_t = -float(np.logaddexp(0.0, -log_y))
        log_one_minus_t = -float(np.logaddexp(0.0, log_y))
        return (log_constant + k * log_t + delta * log_one_minus_t
                + alpha * float(np.logaddexp(log_r1 + log_one_minus_t, log_t))
                + beta * float(np.logaddexp(log_r2 + log_one_minus_t, log_t)))

    values = []
    if k == 0:
        values.append(log_constant + alpha * log_r1 + beta * log_r2)
    if delta == 0.0:
        values.append(log_constant)
    # An identically zero derivative means a constant ratio.
    if coefficient_scale == 0.0:
        return max(values) + _LOG_BOUND_MARGIN
    if len(coefficients) > 1:
        normalized = coefficients / coefficient_scale
        try:
            with np.errstate(all="ignore"):
                roots = np.roots(normalized[::-1])
                reconstructed = np.poly(roots) * normalized[-1]
                if (not np.all(np.isfinite(roots))
                        or not np.all(np.isfinite(reconstructed))
                        or not np.allclose(reconstructed, normalized[::-1],
                                           rtol=1.0e-7, atol=1.0e-10)):
                    return fallback + _LOG_BOUND_MARGIN
                for index, root in enumerate(roots):
                    size = max(1.0, abs(root))
                    # Coalescing roots are where companion eigenvalues can
                    # obscure real stationary points; do not trust that case.
                    for other in roots[index + 1:]:
                        if abs(root - other) < 1.0e-6 * max(size, abs(other)):
                            return fallback + _LOG_BOUND_MARGIN
                    residual = abs(np.polynomial.polynomial.polyval(root, normalized))
                    residual_scale = np.polynomial.polynomial.polyval(abs(root), np.abs(normalized))
                    if (not np.isfinite(residual) or not np.isfinite(residual_scale)
                            or residual > 1.0e-7 * residual_scale):
                        return fallback + _LOG_BOUND_MARGIN
                    if abs(root.imag) <= 1.0e-10 * size:
                        if root.real > 0.0:
                            values.append(log_ratio(math.log(float(root.real))))
                    elif abs(root.imag) < 1.0e-6 * size:
                        return fallback + _LOG_BOUND_MARGIN
        except (FloatingPointError, np.linalg.LinAlgError, ValueError):
            return fallback + _LOG_BOUND_MARGIN
    if not values or not np.all(np.isfinite(values)):
        return fallback + _LOG_BOUND_MARGIN
    maximum = max(values)
    # A candidate exceeding a proven factorwise bound flags numerical trouble.
    if maximum > fallback + 1.0e-7 * max(1.0, abs(fallback)):
        return fallback + _LOG_BOUND_MARGIN
    return maximum + _LOG_BOUND_MARGIN


def log_component_sup(family, a11, a22, ac, nu11=None, nu22=None,
                      nuc=None, p=1, odd=False, d=2):
    """Return log of the continuous-frequency squared spectral-ratio bound.

    Gaussian odd multiplier: u**p. Matérn odd multiplier: (u/ac)**p.
    The normalized angular factor has supremum one. Gaussian even requires
    2*ac**2 >= a11**2+a22**2; odd requires strict inequality. Matérn requires
    2*nuc >= nu11+nu22+(p if odd else 0). Failed restrictions return infinity.
    Invalid marginal scales/smoothness raise ValueError. A zero cross-amplitude
    or an inactive component should be omitted before calling this function.
    """
    if int(d) != d or d < 2:
        raise ValueError("d must be an integer at least two")
    if odd and (int(p) != p or p <= 0 or int(p) % 2 != 1):
        raise ValueError("p must be a positive odd integer")
    a11 = _positive_number(a11, "a11")
    a22 = _positive_number(a22, "a22")
    ac = _positive_number(ac, "ac")
    k = int(p) if odd else 0
    family = str(family).lower()
    if family == "gaussian":
        # Scale first to avoid squaring very large physical scales.
        largest = max(a11, a22, ac)
        scaled_delta = math.fsum([2.0 * (ac / largest) ** 2,
                                 -(a11 / largest) ** 2, -(a22 / largest) ** 2])
        log_delta = None
        if abs(scaled_delta) < 1.0e-12:
            # Close to the equality boundary, a rounded cancellation must not
            # turn a negative Delta into an apparently admissible positive one.
            with localcontext() as context:
                context.prec = 80
                exact_delta = (2 * Decimal.from_float(ac) ** 2
                               - Decimal.from_float(a11) ** 2
                               - Decimal.from_float(a22) ** 2)
                if exact_delta < 0 or (odd and exact_delta == 0):
                    return math.inf
                if exact_delta > 0:
                    log_delta = float(exact_delta.ln())
                    scaled_delta = 1.0  # The sign has been established above.
                else:
                    scaled_delta = 0.0
        if scaled_delta < 0.0 or (odd and scaled_delta == 0.0):
            return math.inf
        answer = 2.0 * d * math.log(ac) - d * (math.log(a11) + math.log(a22))
        if odd:
            if log_delta is None:
                log_delta = math.log(scaled_delta) + 2.0 * math.log(largest)
            answer += k * (math.log(4.0 * k) - log_delta - 1.0)
        return answer + _LOG_BOUND_MARGIN
    if family == "matern":
        nu11 = _positive_number(nu11, "nu11")
        nu22 = _positive_number(nu22, "nu22")
        nuc = _positive_number(nuc, "nuc")
        if odd and nuc <= p / 2.0:
            return math.inf
        return _matern_log_sup(a11, a22, ac, nu11, nu22, nuc, k, d)
    raise ValueError("family must be 'gaussian' or 'matern'")


def mixed_log_sup_bound(log_even, log_odd, theta):
    """Log of cos(theta)^2*M_even + sin(theta)^2*M_odd.

    This is a sufficient upper bound on the mixed spectral supremum. It equals
    the component bound for a pure angle. Inactive terms are omitted before
    inspecting them, so an unused infinity does not cause 0*infinity or NaN.
    Exact multiples of pi/2 are recognized without an arbitrary angle cutoff.
    """
    theta = float(theta)
    if not math.isfinite(theta):
        raise ValueError("theta must be finite")
    phase = math.remainder(theta, math.pi)
    if phase == 0.0:
        return float(log_even)
    if abs(phase) == math.pi / 2.0:
        return float(log_odd)
    even = float(log_even) + 2.0 * math.log(abs(math.cos(phase)))
    odd = float(log_odd) + 2.0 * math.log(abs(math.sin(phase)))
    return float(np.logaddexp(even, odd))

## 3. One data loader for every model

Default CSV columns: `datetime, latitude, longitude, u_std, v_std`.
The loader uses one ordered spatial grid, skips incomplete time slices, and
rejects duplicate time/location rows. It records used/skipped timestamps and a
SHA-256 fingerprint. All models receive exactly the same observations.

Observations remain interleaved `[u1,v1,u2,v2,...]`. The loader does **not**
recenter or restandardize values. By default they are assumed externally
standardized, with known zero means and fixed unit marginal variances, matching
the old workflow. Optional shared means can instead be profiled in the likelihood.

`max_replicates=1` uses the first complete selected slice. Multiple slices are
treated as independent spatial replicates; temporal dependence is not fitted.

All models use one fixed coordinate reference length. The default is 100 km;
change it deliberately and retain it for the whole comparison. Gaussian `a` is
a length; Matérn `a` is an inverse length. Physical-unit conversions are exported.
Lon/lat conversion is the centered equirectangular approximation used by the old
workflow. For more accurate regional geometry, supply projected `xy` columns in km.

In [ ]:
"""Shared CSV loading for every covariance model in the corrected notebook."""
from __future__ import annotations

from dataclasses import dataclass
from hashlib import sha256
from pathlib import Path
from typing import Any, Dict, Optional, Sequence, Tuple, Union
import warnings

import numpy as np
import pandas as pd


@dataclass
class DataConfig:
    path: Union[str, Path] = "Data.csv"
    value_columns: Tuple[str, str] = ("u_std", "v_std")
    time_column: str = "datetime"
    coordinate_columns: Tuple[str, str] = ("latitude", "longitude")
    coordinate_mode: str = "lonlat"
    max_replicates: Optional[int] = 1
    selected_times: Optional[Sequence[str]] = None
    reference_length_km: float = 100.0


@dataclass
class PreparedData:
    locations: np.ndarray
    observations: np.ndarray
    metadata: Dict[str, Any]


def load_csv_data(config: DataConfig) -> PreparedData:
    """Load complete, identically ordered spatial replicates from a CSV.

    Values are neither centered nor standardized. The likelihood uses the
    supplied values as zero-mean observations; externally standardized values
    and fixed marginal standard deviations of one preserve the old workflow.

    The canonical spatial grid is the union of coordinates in the entire CSV,
    including times outside ``selected_times``. A usable replicate must have
    finite values for both variables at every point in that grid. All duplicate
    time/location keys are errors, even if the corresponding time is unselected.

    In ``lonlat`` mode, columns contain (latitude, longitude), in degrees.
    The local equirectangular projection uses 111 km per degree and a circular
    longitude center; it handles a local grid crossing the date line. It is an
    approximation matching the old workflow. A span above 30 degrees produces
    a warning and metadata flag, rather than rejecting the old regional wind
    grid. Near-global spans and polar degeneracy are rejected. For a more
    suitable map projection, project externally and use ``xy`` mode with columns
    (x, y) in km. Every model receives these same numerical locations, obtained
    by dividing km coordinates by the fixed reference length.

    Timestamps are normalized to UTC; naive timestamps are interpreted as UTC.
    Selected times are processed chronologically, and ``max_replicates`` counts
    complete replicates, not merely the first rows or first timestamps.
    """
    if not isinstance(config, DataConfig):
        raise TypeError("config must be a DataConfig instance.")
    if len(config.value_columns) != 2 or len(config.coordinate_columns) != 2:
        raise ValueError("Exactly two value columns and two coordinate columns are required.")
    required = [config.time_column, *config.coordinate_columns, *config.value_columns]
    if len(set(required)) != len(required):
        raise ValueError("Time, coordinate and value column names must all be distinct.")
    if config.coordinate_mode not in {"lonlat", "xy"}:
        raise ValueError("coordinate_mode must be 'lonlat' or 'xy'.")
    length = float(config.reference_length_km)
    if not np.isfinite(length) or length <= 0.0:
        raise ValueError("reference_length_km must be finite and strictly positive.")
    limit = config.max_replicates
    if limit is not None and (
        isinstance(limit, (bool, np.bool_))
        or not isinstance(limit, (int, np.integer))
        or limit < 1
    ):
        raise ValueError("max_replicates must be a positive integer or None for all complete times.")

    path = Path(config.path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(
            "CSV file not found: {}. Set DATA_CONFIG.path to your data file.".format(path.resolve())
        )
    # Hash the bytes actually read so the output identifies this exact dataset.
    file_bytes = path.read_bytes()
    from io import BytesIO
    frame = pd.read_csv(BytesIO(file_bytes))
    missing = [name for name in required if name not in frame.columns]
    if missing:
        raise ValueError("CSV is missing required columns: {}".format(", ".join(missing)))
    if frame.empty:
        raise ValueError("CSV contains no observations.")
    frame = frame.loc[:, required].copy()

    try:
        parsed_times = pd.to_datetime(frame[config.time_column], errors="raise", utc=True)
    except (ValueError, TypeError) as exc:
        raise ValueError("Invalid timestamp in column {!r}.".format(config.time_column)) from exc
    if parsed_times.isna().any():
        raise ValueError("Missing or invalid timestamps are not allowed.")
    frame[config.time_column] = parsed_times
    for name in config.coordinate_columns:
        try:
            frame[name] = pd.to_numeric(frame[name], errors="raise").astype(float)
        except (TypeError, ValueError) as exc:
            raise ValueError("Coordinate column {!r} must contain numeric values.".format(name)) from exc
    if not np.isfinite(frame.loc[:, list(config.coordinate_columns)].to_numpy(dtype=float)).all():
        raise ValueError("Coordinate columns contain missing or nonfinite values.")

    if config.coordinate_mode == "lonlat":
        lat_column, lon_column = config.coordinate_columns
        if (frame[lat_column].abs() > 90.0).any():
            raise ValueError("Latitudes must lie between -90 and 90 degrees.")
        # Treat -180/180 and 0/360 as the same physical longitude when checking
        # duplicates and constructing the canonical grid.
        frame[lon_column] = (frame[lon_column] + 180.0) % 360.0 - 180.0

    key_columns = [config.time_column, *config.coordinate_columns]
    duplicate = frame.duplicated(subset=key_columns, keep=False)
    if duplicate.any():
        example = frame.loc[duplicate, key_columns].iloc[0].to_dict()
        raise ValueError("Duplicate timestamp/location key found: {}. Resolve duplicates before fitting.".format(example))

    coordinate_frame = (
        frame.loc[:, list(config.coordinate_columns)]
        .drop_duplicates()
        .sort_values(list(config.coordinate_columns))
        .reset_index(drop=True)
    )
    n_locations = len(coordinate_frame)
    if n_locations < 2:
        raise ValueError("At least two distinct spatial locations are required.")
    canonical_index = pd.MultiIndex.from_frame(coordinate_frame)
    original_coordinates = coordinate_frame.to_numpy(dtype=float)
    if config.coordinate_mode == "lonlat":
        latitude, longitude = original_coordinates.T
        lat_center = float(latitude.mean())
        radians = np.deg2rad(longitude)
        circular_center = complex(np.cos(radians).mean(), np.sin(radians).mean())
        if abs(circular_center) < 1e-8:
            raise ValueError("Longitude grid is too broad for the local projection; supply projected xy coordinates in km.")
        lon_center = float(np.rad2deg(np.angle(circular_center)))
        longitude_offset = (longitude - lon_center + 180.0) % 360.0 - 180.0
        latitude_span = float(np.ptp(latitude))
        longitude_span = float(np.ptp(longitude_offset))
        if longitude_span >= 180.0 or latitude_span >= 160.0 or abs(lat_center) >= 85.0:
            raise ValueError("Near-global extent or polar degeneracy makes this longitude/latitude projection unsuitable; supply projected xy coordinates in km.")
        broad_extent = latitude_span > 30.0 or longitude_span > 30.0
        projection_warning = None
        if broad_extent:
            projection_warning = (
                "The latitude/longitude grid spans more than 30 degrees. "
                "The equirectangular distances preserve the old notebook's "
                "approximation but can distort separation over this extent. "
                "Use coordinate_mode='xy' with externally projected km "
                "coordinates when a more suitable map projection is available."
            )
            warnings.warn(projection_warning, UserWarning, stacklevel=2)
        locations_km = np.column_stack((
            111.0 * np.cos(np.deg2rad(lat_center)) * longitude_offset,
            111.0 * (latitude - lat_center),
        ))
        projection = {
            "method": "local equirectangular approximation, 111 km/degree",
            "latitude_center_degrees": lat_center,
            "longitude_center_degrees": lon_center,
            "longitude_span_degrees": longitude_span,
            "latitude_span_degrees": latitude_span,
            "broad_extent_approximation": broad_extent,
            "projection_warning": projection_warning,
        }
    else:
        center_km = original_coordinates.mean(axis=0)
        locations_km = original_coordinates - center_km
        projection = {"method": "supplied Cartesian coordinates in km", "center_km": center_km.tolist()}
    locations = np.ascontiguousarray(locations_km / length, dtype=float)
    if not np.isfinite(locations).all():
        raise ValueError("Coordinate projection/scaling produced nonfinite values.")
    if len(np.unique(locations, axis=0)) != n_locations:
        raise ValueError("Coordinate projection/scaling collapsed distinct locations.")

    available_times = pd.DatetimeIndex(frame[config.time_column].unique()).sort_values()
    requested_times = None
    if config.selected_times is not None:
        raw_selection = [config.selected_times] if isinstance(config.selected_times, str) else list(config.selected_times)
        if not raw_selection:
            raise ValueError("selected_times is empty; use None to consider every timestamp.")
        try:
            requested_times = pd.DatetimeIndex(pd.to_datetime(raw_selection, errors="raise", utc=True))
        except (TypeError, ValueError) as exc:
            raise ValueError("selected_times contains an invalid timestamp.") from exc
        if requested_times.isna().any() or requested_times.has_duplicates:
            raise ValueError("selected_times must contain distinct, valid timestamps.")
        absent_times = requested_times.difference(available_times)
        if len(absent_times):
            raise ValueError("Selected timestamps are absent from the CSV: {}".format(", ".join(t.isoformat() for t in absent_times)))
        candidate_times = requested_times.sort_values()
    else:
        candidate_times = available_times

    # Non-numeric value entries make their time slice incomplete; do not replace
    # them with zeros, impute values, or change the common spatial grid.
    for name in config.value_columns:
        frame[name] = pd.to_numeric(frame[name], errors="coerce")
    grouped = frame.groupby(config.time_column, sort=False)
    vectors = []
    used_times = []
    skipped_times = []
    considered_times = []
    for timestamp in candidate_times:
        if limit is not None and len(vectors) >= limit:
            break
        considered_times.append(timestamp.isoformat())
        group = grouped.get_group(timestamp)
        aligned = group.set_index(list(config.coordinate_columns)).reindex(canonical_index)
        values = aligned.loc[:, list(config.value_columns)].to_numpy(dtype=float)
        missing_location_count = n_locations - len(group)
        nonfinite_count = int((~np.isfinite(values)).sum())
        if missing_location_count or nonfinite_count:
            skipped_times.append({
                "timestamp": timestamp.isoformat(),
                "reason": "incomplete spatial grid or nonfinite supplied values",
                "missing_locations": int(missing_location_count),
                "nonfinite_value_entries_after_alignment": nonfinite_count,
            })
            continue
        # Row-major flattening gives [u(s1), v(s1), ..., u(sn), v(sn)].
        vectors.append(values.reshape(-1))
        used_times.append(timestamp.isoformat())
    if not vectors:
        raise ValueError("No complete time replicate found on the {}-location canonical grid. Skipped times: {}".format(n_locations, skipped_times))
    observations = np.ascontiguousarray(np.column_stack(vectors), dtype=float)
    component_values = np.stack((observations[0::2, :], observations[1::2, :]))
    metadata = {
        "data_path": str(path.resolve()),
        "data_sha256": sha256(file_bytes).hexdigest(),
        "row_count": int(len(frame)),
        "n_locations": int(n_locations),
        "n_replicates": int(observations.shape[1]),
        "observation_order": "location-interleaved; columns are time replicates",
        "coordinate_mode": config.coordinate_mode,
        "coordinate_columns": list(config.coordinate_columns),
        "original_coordinate_points": original_coordinates.tolist(),
        "value_columns": list(config.value_columns),
        "time_column": config.time_column,
        "timestamp_convention": "UTC; naive input timestamps interpreted as UTC",
        "reference_length_km": length,
        "projection": projection,
        "canonical_grid": "union of coordinates over the entire CSV, including unselected times",
        "available_time_count": int(len(available_times)),
        "selected_times_requested": None if requested_times is None else [t.isoformat() for t in requested_times],
        "max_replicates_requested": None if limit is None else int(limit),
        "candidate_time_count": int(len(candidate_times)),
        "considered_times": considered_times,
        "used_times": used_times,
        "skipped_times": skipped_times,
        "unexamined_candidate_count": int(len(candidate_times) - len(considered_times)),
        "replicate_limit_satisfied": limit is None or len(vectors) >= limit,
        "values_centered_or_standardized_by_loader": False,
        "supplied_component_means": component_values.mean(axis=(1, 2)).tolist(),
        "supplied_component_standard_deviations_ddof0": component_values.std(axis=(1, 2), ddof=0).tolist(),
        "likelihood_assumptions": "Supplied values have zero mean; retained time replicates are independent under the fitted model.",
    }
    return PreparedData(locations=locations, observations=observations, metadata=metadata)

## 4. Cached geometry and vectorized covariance kernels

Only the radial kernels change between parameter candidates. Distances, exact
unique-distance indices and Chebyshev angular factors are reused. The full
matrix is assembled with `C21 = C12.T`; the odd cross-block is never symmetrized
away. No eigenvalue projection or adaptive diagonal jitter is used.

A nonzero `nugget_variance` is an explicitly specified, fixed observation-noise
variance. It changes the observation model and is recorded in every result.
The default is zero. Failed Cholesky factorizations are rejected and counted.

In [ ]:
class Geometry:
    """Cached exact distances/angles; C[i,j] uses h = location[j]-location[i]."""

    def __init__(self, locations, direction_degrees=0.0):
        loc = np.asarray(locations, dtype=float)
        if loc.ndim != 2 or loc.shape[1] != 2 or len(loc) < 2 or not np.isfinite(loc).all():
            raise ValueError('locations must be a finite (N,2) array with N>=2.')
        if len(np.unique(loc, axis=0)) != len(loc):
            raise ValueError('Duplicate sites are not supported; aggregate or model replicate errors explicitly.')
        if not np.isfinite(direction_degrees):
            raise ValueError('direction_degrees must be finite.')
        self.locations = loc.copy()
        self.n = len(loc)
        self.d = 2
        angle = np.deg2rad(direction_degrees)
        self.u0 = np.array([np.cos(angle), np.sin(angle)])
        lag = loc[None, :, :] - loc[:, None, :]
        self.r2 = np.einsum('ijk,ijk->ij', lag, lag)
        self.r = np.sqrt(self.r2)
        self.unique_r, inverse = np.unique(self.r, return_inverse=True)
        self.radial_index = inverse.reshape(self.r.shape)
        cosine = np.zeros_like(self.r)
        np.divide(lag @ self.u0, self.r, out=cosine, where=self.r > 0)
        self.cosine = np.clip(cosine, -1.0, 1.0)
        self.design = np.tile(np.eye(2), (self.n, 1))
        self._angular = {}
        self.direction_degrees = float(direction_degrees)

    def angular(self, p):
        if p not in self._angular:
            if p < 1 or p % 2 != 1:
                raise ValueError('p must be positive and odd.')
            a = eval_chebyt(p, self.cosine)
            # Enforce the exact known parity against floating-point evaluation noise.
            a = 0.5 * (a - a.T)
            np.fill_diagonal(a, 0.0)
            self._angular[p] = a
        return self._angular[p]

    def expand(self, values):
        return np.asarray(values)[self.radial_index]


def gaussian_radial(r, a, p=None):
    r = np.asarray(r, dtype=float)
    if not np.isfinite(a) or a <= 0 or np.any(r < 0) or not np.isfinite(r).all():
        raise ValueError('Gaussian radius/scale outside its domain.')
    if p is None:
        return np.exp(-(r / a)**2)
    if p < 1 or p % 2 != 1:
        raise ValueError('Odd degree required.')
    out = np.zeros_like(r)
    positive = r > 0
    logs = p * (math.log(2.0) - 2*math.log(a) + np.log(r[positive])) - (r[positive]/a)**2
    out[positive] = ((-1)**((p-1)//2)) * np.exp(logs)
    return out


def matern_radial(r, a, nu, p=None):
    """Normalized even or odd radial inverse; exact zero-lag assignments.

    log(kve(order,z))-z avoids large-z underflow in the intermediate K.
    A small-z leading expansion is used ONLY if the special function overflows
    and its mathematical remainder is negligible in that regime. Other numerical
    failures are rejected, never replaced with zero covariance.
    """
    r = np.asarray(r, dtype=float)
    if a <= 0 or nu <= 0 or not np.isfinite([a, nu]).all() or np.any(r < 0) or not np.isfinite(r).all():
        raise ValueError('Matern radius/scale/smoothness outside its domain.')
    if p is not None and (p < 1 or p % 2 != 1 or nu <= p/2):
        raise ValueError('Odd Matern requires positive odd p and nu>p/2.')
    out = np.zeros_like(r) if p is not None else np.ones_like(r)
    positive = r > 0
    z = a*r[positive]
    order = nu if p is None else abs(p-nu)
    with np.errstate(over='ignore', under='ignore', divide='ignore', invalid='ignore'):
        logk = np.log(kve(order, z)) - z
    bad = ~np.isfinite(logk)
    if bad.any():
        recoverable = bad & (z < 1e-8) & (order > 1e-4)
        logk[recoverable] = gammaln(order)+(order-1)*math.log(2.0)-order*np.log(z[recoverable])
        if not np.isfinite(logk).all():
            raise FloatingPointError('Bessel evaluation failed; tighten parameter bounds or inspect coordinate units.')
    logs = (1-nu)*math.log(2.0)-gammaln(nu)+nu*np.log(z)+logk
    values = np.exp(logs)
    if not np.isfinite(values).all():
        raise FloatingPointError('Nonfinite normalized Matern covariance.')
    if p is not None:
        values *= (-1)**((p-1)//2)
    else:
        if np.any(values > 1.0+1e-9):
            raise FloatingPointError('Matern numerical evaluation exceeds its unit sill.')
        values = np.minimum(values, 1.0)
    out[positive] = values
    return out


def covariance_matrix(spec, params, geometry, sds=(1.0, 1.0), nugget_variance=0.0):
    """Assemble a 2N x 2N interleaved matrix; no eigendecomposition/projection."""
    radial = geometry.unique_r
    if spec.family == 'G':
        k11 = geometry.expand(gaussian_radial(radial, params['a11']))
        k22 = geometry.expand(gaussian_radial(radial, params['a22']))
    else:
        k11 = geometry.expand(matern_radial(radial, params['a11'], params['nu11']))
        k22 = geometry.expand(matern_radial(radial, params['a22'], params['nu22']))
    cross = np.zeros_like(k11)
    rho = params['rho']
    if spec.has_even and params['weight_even'] != 0 and rho != 0:
        e = (gaussian_radial(radial, params['a_even']) if spec.family == 'G'
             else matern_radial(radial, params['a_even'], params['nu_even']))
        cross += rho*params['weight_even']*geometry.expand(e)
    if spec.has_odd and params['weight_odd'] != 0 and rho != 0:
        o = (gaussian_radial(radial, params['a_odd'], spec.p) if spec.family == 'G'
             else matern_radial(radial, params['a_odd'], params['nu_odd'], spec.p))
        cross += rho*params['weight_odd']*geometry.expand(o)*geometry.angular(spec.p)
    c = np.empty((2*geometry.n, 2*geometry.n), dtype=float)
    c[0::2, 0::2] = sds[0]**2*k11
    c[1::2, 1::2] = sds[1]**2*k22
    c[0::2, 1::2] = sds[0]*sds[1]*cross
    c[1::2, 0::2] = sds[0]*sds[1]*cross.T
    if nugget_variance:
        c.flat[::c.shape[0]+1] += nugget_variance
    if not np.isfinite(c).all():
        raise FloatingPointError('Nonfinite covariance matrix.')
    return c

## 5. Named parameters and valid search coordinates

Log length parameters improve scaling across orders of magnitude. Gaussian
cross-scales satisfy the appropriate even/non-strict and odd/strict inequalities.
Matérn cross-smoothness is parameterized as its spectral tail threshold plus a
nonnegative gap. Bounds are adjustable in `FitConfig` below.

The optimizer's `rho_fraction` measures proximity to the spectral bound;
`params["rho"]` is the manuscript's actual amplitude. A signed amplitude and
`theta` in `[0, pi]` cover the full even/odd coefficient plane without the old
redundancy between `(rho,theta)` and `(-rho,theta+pi)`.

In [ ]:
@dataclass(frozen=True)
class ModelSpec:
    family: str
    kind: str
    p: int = 1

    def __post_init__(self):
        if self.family not in ('G', 'M') or self.kind not in ('even', 'odd', 'mixed'):
            raise ValueError('Use family G/M and kind even/odd/mixed.')
        if isinstance(self.p,(bool,np.bool_)) or not isinstance(self.p, (int, np.integer)) or self.p < 1 or self.p % 2 != 1:
            raise ValueError('p must be a positive odd integer.')

    @property
    def name(self):
        return self.family + '_' + self.kind

    @property
    def has_even(self):
        return self.kind != 'odd'

    @property
    def has_odd(self):
        return self.kind != 'even'


@dataclass
class FitConfig:
    seed: int = 1729
    n_candidates: int = 48
    n_starts: int = 3
    maxiter: int = 10000
    ftol: float = 1e-8
    # Lengths are in the common numerical reference unit; None derives bounds
    # from the minimum spacing and domain diameter, identically for all models.
    length_bounds: tuple = None
    marginal_nu_bounds: tuple = (0.15, 600.0)
    cross_nu_gap_max: float = 600.0
    gaussian_gap_max: float = 20.0
    gaussian_odd_gap_min: float = 1e-3
    fraction_limit: float = 0.995
    mean_mode: str = 'zero'       # 'zero' matches the original; 'profile' fits 2 means
    marginal_sds: tuple = (1.0, 1.0)
    nugget_variance: float = 0.0  # explicit, fixed observation-noise variance
    bic_count: str = 'scalar'    # 2*N*R, or 'locations' (N*R), or 'replicates' (R)
    blas_threads: int = 1
    fixed_smoothness: dict = field(default_factory=dict)
    # fixed_smoothness keys: nu11, nu22, nu_even, nu_odd. Cross values must
    # satisfy the smoothness tail restriction for each fitted p.

    def validate(self):
        if any(isinstance(v,(bool,np.bool_)) or not isinstance(v,(int,np.integer)) or v < 1
               for v in (self.n_candidates,self.n_starts,self.maxiter,self.blas_threads)):
            raise ValueError('Candidate/start/iteration counts must be positive.')
        if not (0 < self.fraction_limit < 1):
            raise ValueError('fraction_limit must lie strictly between 0 and 1.')
        if self.mean_mode not in ('zero', 'profile'):
            raise ValueError('mean_mode must be zero or profile.')
        if self.bic_count not in ('scalar', 'locations', 'replicates'):
            raise ValueError('Unknown BIC count convention.')
        if len(self.marginal_sds) != 2 or min(self.marginal_sds) <= 0:
            raise ValueError('Two positive marginal standard deviations are required.')
        if not np.all(np.isfinite(self.marginal_sds)):
            raise ValueError('Marginal standard deviations must be finite.')
        if not np.isfinite(self.nugget_variance) or self.nugget_variance < 0:
            raise ValueError('nugget_variance must be finite and nonnegative.')
        if self.cross_nu_gap_max <= 0 or self.gaussian_gap_max <= 0:
            raise ValueError('Cross-gap upper bounds must be positive.')
        if not 0 < self.gaussian_odd_gap_min < self.gaussian_gap_max:
            raise ValueError('Gaussian odd gap bounds must be positive and ordered.')
        lo, hi = self.marginal_nu_bounds
        if not 0 < lo < hi:
            raise ValueError('Smoothness bounds must be positive and ordered.')
        allowed = {'nu11', 'nu22', 'nu_even', 'nu_odd'}
        if set(self.fixed_smoothness) - allowed:
            raise ValueError('Unknown fixed_smoothness parameter.')
        if any(not np.isfinite(v) or v <= 0 for v in self.fixed_smoothness.values()):
            raise ValueError('Fixed smoothness values must be positive and finite.')

In [ ]:
class ParameterSpace:
    """Named free parameters mapped to a spectrally admissible model.

    Pure components use continuous-frequency maxima. Mixed models use the
    sufficient sum of component maxima, which can be conservative. rho_fraction
    is bounded; the actual odd amplitude rho is NOT restricted to [-1,1].
    """
    def __init__(self, spec, geometry, config):
        config.validate()
        self.spec, self.geometry, self.config = spec, geometry, config
        positive = geometry.unique_r[geometry.unique_r > 0]
        lo, hi = config.length_bounds or (max(0.25*positive.min(), 1e-4), 3*positive.max())
        if not 0 < lo < hi or not np.isfinite([lo, hi]).all():
            raise ValueError('length_bounds must be finite, positive and increasing.')
        self.names, self.bounds = [], []
        self.add('log_length11', (np.log(lo), np.log(hi)))
        self.add('log_length22', (np.log(lo), np.log(hi)))
        if spec.family == 'G':
            if spec.has_even:
                self.add('gap_even', (0.0, config.gaussian_gap_max))
            if spec.has_odd:
                self.add('log_gap_odd', (np.log(config.gaussian_odd_gap_min), np.log(config.gaussian_gap_max)))
        else:
            if spec.has_even:
                self.add('log_length_even', (np.log(lo), np.log(hi)))
            if spec.has_odd:
                self.add('log_length_odd', (np.log(lo), np.log(hi)))
            for key in ('nu11', 'nu22'):
                if key not in config.fixed_smoothness:
                    self.add(key, config.marginal_nu_bounds)
            for part in ('even', 'odd'):
                active = spec.has_even if part == 'even' else spec.has_odd
                if active and 'nu_'+part not in config.fixed_smoothness:
                    self.add('nu_gap_'+part, (0.0, config.cross_nu_gap_max))
        if spec.kind == 'mixed':
            # Signed rho plus theta in [0,pi] covers the full coefficient plane.
            self.add('theta', (0.0, np.pi))
        self.add('rho_fraction', (-config.fraction_limit, config.fraction_limit))
        self.bounds = np.asarray(self.bounds, dtype=float)

    def add(self, name, bounds):
        self.names.append(name)
        self.bounds.append(bounds)

    def decode(self, x):
        x = np.asarray(x, dtype=float)
        if x.shape != (len(self.names),) or not np.isfinite(x).all():
            raise ValueError('Invalid optimizer parameter vector.')
        if np.any(x < self.bounds[:,0]) or np.any(x > self.bounds[:,1]):
            raise ValueError('Optimizer parameter vector is outside the configured bounds.')
        z = dict(zip(self.names, x))
        spec, cfg = self.spec, self.config
        theta = float(z.get('theta', 0.0 if spec.kind == 'even' else np.pi/2))
        # Treat the mathematical endpoints exactly, including floating pi/2.
        we = 0.0 if spec.kind == 'odd' or theta == np.pi/2 else float(np.cos(theta))
        wo = 0.0 if spec.kind == 'even' or theta in (0.0, np.pi) else float(np.sin(theta))
        a11, a22 = np.exp(z['log_length11']), np.exp(z['log_length22'])
        if spec.family == 'M':
            a11, a22 = 1/a11, 1/a22
        par = {'a11':float(a11), 'a22':float(a22)}
        if spec.family == 'G':
            base = 0.5*(a11*a11+a22*a22)
            if spec.has_even:
                par['a_even'] = float(np.nextafter(np.sqrt(base*(1+z['gap_even'])), np.inf))
            if spec.has_odd:
                par['a_odd'] = float(np.sqrt(base*(1+np.exp(z['log_gap_odd']))))
        else:
            for key in ('nu11', 'nu22'):
                par[key] = float(cfg.fixed_smoothness[key] if key in cfg.fixed_smoothness else z[key])
            avg = (par['nu11']+par['nu22'])/2
            for part in ('even', 'odd'):
                active = spec.has_even if part == 'even' else spec.has_odd
                if active:
                    par['a_'+part] = float(np.exp(-z['log_length_'+part]))
                    minimum = avg+(spec.p/2 if part == 'odd' else 0)
                    key = 'nu_'+part
                    par[key] = float(cfg.fixed_smoothness[key] if key in cfg.fixed_smoothness
                                     else np.nextafter(minimum+z['nu_gap_'+part], np.inf))
                    weight = we if part == 'even' else wo
                    if weight != 0 and par[key] < minimum:
                        raise ValueError('Fixed cross-smoothness violates the spectral tail condition.')
        logs = {}
        for part, active, weight in [('even', spec.has_even, we), ('odd', spec.has_odd, wo)]:
            if active and weight != 0:
                logs[part] = log_component_sup({'G':'gaussian','M':'matern'}[spec.family], a11, a22, par['a_'+part],
                    nu11=par.get('nu11'), nu22=par.get('nu22'), nuc=par.get('nu_'+part),
                    p=spec.p, odd=(part == 'odd'), d=2)
        if spec.kind == 'even':
            log_bound = logs['even']
        elif spec.kind == 'odd':
            log_bound = logs['odd']
        else:
            terms = [2*np.log(abs(w))+logs[k] for k,w in [('even',we),('odd',wo)] if w != 0]
            log_bound = float(np.logaddexp.reduce(terms))
        if not np.isfinite(log_bound):
            raise ValueError('No finite spectral amplitude bound for these parameters.')
        rho_max = float(np.exp(-0.5*log_bound))
        if not np.isfinite(rho_max) or rho_max <= 0:
            raise FloatingPointError('Amplitude bound cannot be represented at these parameter scales.')
        par.update(rho_fraction=float(z['rho_fraction']), rho_max=rho_max,
                   rho=float(z['rho_fraction']*rho_max), theta=theta,
                   weight_even=we, weight_odd=wo, log_spectral_bound=log_bound,
                   spectral_ratio_bound=float(z['rho_fraction']**2))
        if spec.family == 'M' and spec.has_odd and wo != 0:
            nu_star = par['nu_odd']-spec.p/2
            # b_2(nu)=Gamma(nu+1)/Gamma(nu)=nu.
            par['nu_star'] = nu_star
            par['rho_eff'] = par['rho']*par['nu_odd']/nu_star
        return par

    def vector_from_named(self, values):
        x = self.bounds.mean(axis=1)
        for i, key in enumerate(self.names):
            if key in values:
                x[i] = np.clip(values[key], *self.bounds[i])
        return x

## 6. Likelihood, multistarts and saved comparisons

For $R$ independent replicates, the Gaussian negative log-likelihood is
$\tfrac12\{\sum_r y_r^T C^{-1}y_r+R\log|C|+2NR\log(2\pi)\}$.
All right-hand sides share one triangular solve. Optional means are profiled
by GLS and counted as two fitted parameters. This is ML, not REML.

Initialization uses a reproducible Latin-hypercube design and multiple local
starts, plus warm starts. The best finite likelihood is retained even if a
later optimizer fails. Such a result is explicitly marked **unconverged**,
not silently accepted as a successful MLE. All failures and boundary hits are saved.

With fixed means, sills and direction, parameter counts are 4/6 for Gaussian
pure/mixed models and 7/10 for Matérn pure/mixed models. Fixing smoothness reduces
these counts; profiling means adds two. The default BIC count is $2NR$ to match
the old notebook, explicitly as a convention rather than a claim of independent
scalar observations. Alternative conventions are configurable. AIC/BIC are
conditional on each chosen $p$, direction and preprocessing; searching these
choices adds selection uncertainty not captured by the displayed penalties.

In [ ]:
class LikelihoodProblem:
    """One factorization and one batched triangular solve per candidate."""
    def __init__(self, space, observations):
        self.space = space
        y = np.asarray(observations, dtype=float)
        if y.ndim == 1:
            y = y[:, None]
        if y.ndim != 2 or y.shape[0] != 2*space.geometry.n or y.shape[1] < 1 or not np.isfinite(y).all():
            raise ValueError('observations must be finite and interleaved, shape (2*N,R).')
        self.y = np.asfortranarray(y)
        self.evaluations = 0
        self.failures = {}
        self.best = None
        self._cached = lru_cache(maxsize=512)(self._evaluate)

    def _evaluate(self, key):
        self.evaluations += 1
        try:
            cfg, geo = self.space.config, self.space.geometry
            par = self.space.decode(key)
            c = covariance_matrix(self.space.spec, par, geo, cfg.marginal_sds, cfg.nugget_variance)
            factor = cholesky(c, lower=True, check_finite=False, overwrite_a=True)
            if cfg.mean_mode == 'profile':
                rhs = np.column_stack([self.y, geo.design])
                white = solve_triangular(factor, rhs, lower=True, check_finite=False)
                wy, wx = white[:, :self.y.shape[1]], white[:, self.y.shape[1]:]
                beta = np.linalg.solve(wx.T@wx, wx.T@wy.mean(axis=1))
                resid = wy-wx@beta[:, None]
            else:
                beta = np.zeros(2)
                resid = solve_triangular(factor, self.y, lower=True, check_finite=False)
            m, reps = self.y.shape
            nll = 0.5*(np.sum(resid*resid)+reps*(2*np.log(np.diag(factor)).sum()+m*np.log(2*np.pi)))
            if not np.isfinite(nll):
                raise FloatingPointError('Nonfinite likelihood.')
            result = (float(nll), par, beta.tolist())
            if self.best is None or nll < self.best['nll']:
                self.best = dict(nll=float(nll), x=np.array(key), params=par, means=beta.tolist())
            return result
        except (ValueError, FloatingPointError, np.linalg.LinAlgError, OverflowError) as exc:
            reason = type(exc).__name__+': '+str(exc)
            self.failures[reason] = self.failures.get(reason, 0)+1
            return (np.inf, None, None)

    def evaluate(self, x):
        return self._cached(tuple(np.asarray(x, dtype=float)))

    def objective(self, x):
        # Finite penalty keeps finite differences defined at rejected candidates;
        # only genuinely finite likelihoods may become selected results.
        val = self.evaluate(x)[0]
        return val if np.isfinite(val) else 1e50


def fit_model(spec, geometry, observations, config=None, warm_starts=None):
    config = config or FitConfig()
    started = time.perf_counter()
    space = ParameterSpace(spec, geometry, config)
    problem = LikelihoodProblem(space, observations)
    try:
        from threadpoolctl import threadpool_limits
        threads = threadpool_limits(limits=config.blas_threads, user_api='blas')
    except ImportError:
        threads = nullcontext()
    except (OSError, RuntimeError, AttributeError) as exc:
        warnings.warn('Optional BLAS thread control is unavailable in this kernel; '
                      'using the existing thread settings. ' + str(exc), RuntimeWarning)
        threads = nullcontext()
    optimizer_runs = []
    with threads:
        # seed works with SciPy 1.10 (the original notebook environment) onward.
        sample = qmc.LatinHypercube(d=len(space.names), seed=config.seed).random(config.n_candidates)
        candidates = space.bounds[:,0]+sample*(space.bounds[:,1]-space.bounds[:,0])
        informed = {'rho_fraction':0.0, 'theta':np.pi/4,
                    'gap_even':0.25, 'log_gap_odd':0.0,
                    'nu11':1.5, 'nu22':1.5, 'nu_gap_even':0.5, 'nu_gap_odd':0.5}
        extra = [space.vector_from_named(informed)]
        extra += [space.vector_from_named(w) for w in (warm_starts or [])]
        candidates = np.vstack(extra+[candidates])
        scores = np.array([problem.evaluate(x)[0] for x in candidates])
        finite = np.flatnonzero(np.isfinite(scores))
        selected = finite[np.argsort(scores[finite])[:config.n_starts]]
        for start_i, idx in enumerate(selected):
            res = minimize(problem.objective, candidates[idx], method='L-BFGS-B',
                           bounds=space.bounds, options={'maxiter':config.maxiter,
                           'ftol':config.ftol, 'maxls':30})
            val = problem.evaluate(res.x)[0]
            optimizer_runs.append({'start':int(start_i), 'success':bool(res.success),
                'message':str(res.message), 'nll':float(val), 'nit':int(res.nit),
                'nfev':int(res.nfev), 'x':res.x.tolist()})
    k = len(space.names)+(2 if config.mean_mode == 'profile' else 0)
    reps = problem.y.shape[1]
    bic_n = {'scalar':2*geometry.n*reps, 'locations':geometry.n*reps, 'replicates':reps}[config.bic_count]
    common = {'model':spec.name, 'p':spec.p if spec.has_odd else None, 'k':k,
        'N':geometry.n, 'R':reps, 'bic_count':config.bic_count, 'bic_n':bic_n,
        'seconds':time.perf_counter()-started, 'evaluations':problem.evaluations,
        'optimizer_runs':optimizer_runs, 'evaluation_failures':problem.failures,
        'validity_method':'continuous-frequency component maxima; sufficient mixed bound',
        'mean_mode':config.mean_mode, 'nugget_variance':config.nugget_variance,
        'marginal_sds':list(config.marginal_sds)}
    if problem.best is None:
        common.update(status='failed', converged=False, nll=None, aic=None, bic=None,
                      params={}, free_parameters={}, fitted_means=None, bounds_reached=[])
        return common
    best = problem.best
    converged = any(r['success'] and np.isfinite(r['nll']) and
                    abs(r['nll']-best['nll']) <= max(1e-5, config.ftol*abs(best['nll'])*10)
                    for r in optimizer_runs)
    distances = np.minimum(best['x']-space.bounds[:,0], space.bounds[:,1]-best['x'])
    reached = [name for name,gap,width in zip(space.names, distances, np.ptp(space.bounds,axis=1))
               if gap <= 1e-4*max(width,1.0)]
    common.update(status='converged' if converged else 'best_finite_unconverged',
        converged=converged, nll=best['nll'], aic=2*best['nll']+2*k,
        bic=2*best['nll']+k*np.log(bic_n), params=best['params'],
        free_parameters=dict(zip(space.names, best['x'].tolist())),
        fitted_means=best['means'], bounds_reached=reached)
    return common

In [ ]:
from __future__ import annotations

from dataclasses import asdict, replace
from datetime import datetime, timezone
from pathlib import Path
import json
import platform
import hashlib
import numpy as np
import pandas as pd
import scipy


def _json_safe(value):
    if isinstance(value, dict):
        return {str(k):_json_safe(v) for k,v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [_json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, Path):
        return str(value)
    return value


def comparison_table(results, reference_length_km=100.0):
    rows = []
    for result in results:
        keep = ('model','p','status','converged','nll','aic','bic','k','N','R',
                'bic_n','bic_count','seconds','evaluations','mean_mode','nugget_variance')
        row = {key:result.get(key) for key in keep}
        par = result.get('params', {})
        row.update(par)
        row['bounds_reached'] = '; '.join(result.get('bounds_reached', []))
        means = result.get('fitted_means')
        row['mean1'], row['mean2'] = (means if means is not None else [np.nan,np.nan])
        for key in ('a11','a22','a_even','a_odd'):
            if key in par:
                if result['model'].startswith('G'):
                    row[key+'_km'] = par[key]*reference_length_km
                else:
                    row[key+'_per_km'] = par[key]/reference_length_km
                    row[key+'_reciprocal_km'] = reference_length_km/par[key]
        if par:
            row['latent_collocated_correlation'] = par['rho']*par['weight_even']
            s1,s2 = result['marginal_sds']
            nugget = result['nugget_variance']
            row['observed_collocated_correlation'] = row['latent_collocated_correlation']*s1*s2/np.sqrt(
                (s1*s1+nugget)*(s2*s2+nugget))
            row['even_amplitude'] = par['rho']*par['weight_even']
            # An unscaled u^p Gaussian odd multiplier changes its units with p.
            row['odd_amplitude_km_power_p'] = par['rho']*par['weight_odd']*(
                reference_length_km**(result['p'] or 1) if result['model'].startswith('G') else 1.0)
            row['odd_amplitude_units'] = 'km^p' if result['model'].startswith('G') else 'dimensionless'
        rows.append(row)
    return pd.DataFrame(rows)


def _atomic_text(path, content):
    path = Path(path)
    temporary = path.with_name(path.name+'.tmp')
    temporary.write_text(content, encoding='utf-8')
    temporary.replace(path)


def save_checkpoint(directory, results, metadata):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    table = comparison_table(results, metadata.get('reference_length_km',100.0))
    _atomic_text(directory/'fits.csv', table.to_csv(index=False))
    _atomic_text(directory/'fits.json', json.dumps(_json_safe(results), indent=2, allow_nan=False))
    _atomic_text(directory/'run_metadata.json', json.dumps(_json_safe(metadata), indent=2, allow_nan=False))
    runs = []
    for result in results:
        for attempt in result.get('optimizer_runs',[]):
            runs.append(dict(model=result['model'],p=result['p'],**attempt))
    _atomic_text(directory/'optimizer_runs.csv', pd.DataFrame(runs).to_csv(index=False))


def fit_suite(data, direction_degrees=0.0, p_values=(1,3,5), config=None,
              output_directory=None, model_names=None, verbose=True):
    """Fit each even model once and each odd/mixed model at every requested p.

    Checkpoints include failures and unconverged fits. There is no outlier
    removal and no reuse of numerical estimates from the old notebook.
    """
    config = config or FitConfig()
    p_values = tuple(p_values)
    if not p_values or any(isinstance(p,(bool,np.bool_)) or not isinstance(p,(int,np.integer))
                           or p < 1 or p % 2 != 1 for p in p_values):
        raise ValueError('Provide one or more positive odd p values.')
    p_values = tuple(dict.fromkeys(int(p) for p in p_values))
    model_names = tuple(model_names or ('G_even','G_odd','G_mixed','M_even','M_odd','M_mixed'))
    allowed = {'G_even','G_odd','G_mixed','M_even','M_odd','M_mixed'}
    if set(model_names)-allowed or len(set(model_names)) != len(model_names):
        raise ValueError('model_names must be distinct supported model names.')
    geometry = Geometry(data.locations, direction_degrees)
    results, previous = [], {}
    metadata = dict(data.metadata)
    metadata.update(fit_config=asdict(config), direction_degrees=direction_degrees,
                    p_candidates=list(p_values), model_names=list(model_names),
                    python=platform.python_version(), numpy=np.__version__, scipy=scipy.__version__,
                    pandas=pd.__version__, engine_version='corrected-spectral-mle-1.0',
                    likelihood='Gaussian ML; independent spatial replicates if R>1',
                    bic_note='Configured count convention, not a claim of scalar independence.',
                    validity='Analytic continuous-frequency component bounds; sufficient mixed subset.')
    if output_directory is not None:
        directory = Path(output_directory)
        directory.mkdir(parents=True, exist_ok=True)
        # Refuse to overwrite another fit run; the notebook supplies a new timestamp.
        if (directory/'run_metadata.json').exists():
            raise FileExistsError('Choose a new output_directory; a saved run already exists here.')
        save_checkpoint(directory, results, metadata)
    for name in model_names:
        family, kind = name.split('_')
        for degree in ((1,) if kind == 'even' else p_values):
            spec = ModelSpec(family,kind,degree)
            warm = []
            if name in previous:
                warm.append(previous[name])
            if kind == 'mixed':
                for endpoint, theta in [('even',0.0),('odd',np.pi/2)]:
                    endpoint_key = (family+'_'+endpoint, 1 if endpoint == 'even' else degree)
                    if endpoint_key in previous:
                        warm.append(dict(previous[endpoint_key],theta=theta))
            cfg = replace(config,seed=config.seed+len(results))
            if verbose:
                print('Fitting {}{} ...'.format(name, ' p='+str(degree) if spec.has_odd else ''),flush=True)
            result = fit_model(spec,geometry,data.observations,cfg,warm)
            results.append(result)
            if result['free_parameters']:
                previous[name] = result['free_parameters']
                previous[(name,degree)] = result['free_parameters']
            if output_directory is not None:
                save_checkpoint(directory,results,metadata)
            if verbose:
                print('  {} | nLL={} | {:.2f}s | {} evaluations'.format(
                    result['status'], result['nll'],result['seconds'],result['evaluations']),flush=True)
    return results, comparison_table(results,metadata.get('reference_length_km',100.0))


def selected_fits(table, criterion='aic', converged_only=True):
    """Conditional p selection; failed or unverified estimates are not silently ranked."""
    if criterion not in ('nll','aic','bic'):
        raise ValueError('criterion must be nll, aic or bic.')
    valid = table[np.isfinite(pd.to_numeric(table[criterion],errors='coerce'))].copy()
    if converged_only:
        valid = valid[valid['converged']]
    return valid.sort_values(criterion).groupby('model',sort=False).head(1).reset_index(drop=True)


def plot_model_comparison(table):
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print('Matplotlib is unavailable; saved fit tables remain available. Skipping the comparison plot.')
        return None
    valid = table[np.isfinite(pd.to_numeric(table['aic'],errors='coerce'))].copy()
    if valid.empty:
        print('No finite model fits to plot.')
        return None
    labels = [row.model+(' p='+str(int(row.p)) if pd.notna(row.p) else '')
              for row in valid.itertuples()]
    colors = ['#246d8b' if c else '#cf813d' for c in valid['converged']]
    fig, ax = plt.subplots(figsize=(max(8,len(valid)*0.7),4))
    ax.bar(np.arange(len(valid)),valid['aic']-valid['aic'].min(),color=colors)
    ax.set_xticks(np.arange(len(valid)))
    ax.set_xticklabels(labels,rotation=45,ha='right')
    ax.set_ylabel('AIC minus smallest finite AIC')
    ax.set_title('Conditional model comparison (orange: optimizer did not converge)')
    fig.tight_layout()
    return fig


def synthetic_example(seed=731, n_side=4, replicates=2):
    """Small labelled synthetic dataset for tests/demonstration, never a CSV fallback."""
    grid = np.linspace(0,3,n_side)
    xx,yy = np.meshgrid(grid,grid)
    loc = np.column_stack([xx.ravel(),yy.ravel()])
    geo = Geometry(loc,20.0)
    spec = ModelSpec('M','mixed',1)
    cfg = FitConfig(length_bounds=(0.3,5.0))
    space = ParameterSpace(spec,geo,cfg)
    truth = space.vector_from_named(dict(log_length11=0.0,log_length22=np.log(1.3),
        log_length_even=0.0,log_length_odd=0.0,nu11=1.5,nu22=2.5,
        nu_gap_even=0.3,nu_gap_odd=0.3,theta=0.8,rho_fraction=0.65))
    params = space.decode(truth)
    cov = covariance_matrix(spec,params,geo)
    rng = np.random.default_rng(seed)
    observations = cholesky(cov,lower=True)@rng.standard_normal((2*len(loc),replicates))
    data = PreparedData(loc,observations,dict(source='synthetic demonstration only',
        reference_length_km=100.0,seed=seed,N=len(loc),R=replicates))
    return data,params

## 7. Configuration — edit this cell

For wind, use `uv_wind_multi_for_C2.csv` and the chosen direction (the manuscript
uses 0 degrees). For CAMS, set its CSV path and column names, with the chosen
direction (the manuscript uses 90 degrees). The old notebook used several
inconsistent directions; this setting is shared across the full comparison.

Start with fewer candidates/starts for an exploratory run. Increase them for
the final analysis and investigate bound hits or unconverged results.
`fixed_smoothness` can fix, for example, `nu11` and `nu22`; fixed cross values
`nu_even`/`nu_odd` must also satisfy the tail condition for every fitted degree.

In [ ]:
DATA_CONFIG = DataConfig(
    path=r"D:\Users\meazz\Desktop\mle\uv_wind_multi_for_C2.csv",#uv_wind_multi_for_C2.csv",  # Set an absolute path if the CSV is elsewhere.
    value_columns=("u_std", "v_std"),
    coordinate_columns=("latitude", "longitude"),
    coordinate_mode="lonlat",  # or "xy" with columns (x_km, y_km)
    max_replicates=1,
    selected_times=["2000-01-09 00:00:00"],#["2000-01-02 12:00:00"],#,#None,       # e.g. ["2025-07-01 00:00:00"]
    reference_length_km=100.0,
)
DIRECTION_DEGREES = 90#0.0
P_VALUES = (1, 3, 5, 7)            # any positive odd integers, including 7
MODEL_NAMES = ("G_even", "G_odd", "G_mixed", "M_even", "M_odd", "M_mixed")
FIT_CONFIG = FitConfig(
    seed=1729, n_candidates=48, n_starts=3, maxiter=10000,
    mean_mode="zero",          # "profile" estimates two shared component means
    marginal_sds=(1.0, 1.0),
    nugget_variance=0.0,
    bic_count="scalar",        # old convention: 2*N*R
    fixed_smoothness={},       # e.g. {"nu11": 1.5, "nu22": 2.5}
)
RUN_FITS = True
RUN_SYNTHETIC_DEMO = False
OUTPUT_ROOT = Path("mle_results")

## 8. Load once, inspect, then fit

Missing data never trigger a synthetic fallback. Set `RUN_FITS=False` if you
want to inspect the selected data before fitting. Each run writes a new folder
containing `fits.csv`, full `fits.json`, `optimizer_runs.csv` and
`run_metadata.json`. Checkpoints are updated after each model/degree.

In [ ]:
data = None
results = []
comparison = pd.DataFrame()
run_directory = None
if Path(DATA_CONFIG.path).expanduser().is_file():
    data = load_csv_data(DATA_CONFIG)
    print("Locations:", data.locations.shape[0], "  Complete replicates:", data.observations.shape[1])
    print(json.dumps(_json_safe(data.metadata), indent=2))
else:
    print("No CSV loaded. Set DATA_CONFIG.path to your data file; no real-data fits have run.")

In [ ]:
if RUN_FITS and data is not None:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    run_directory = OUTPUT_ROOT / ("run_" + stamp)
    data.metadata["notebook_code_sha256"] = "3cfd097706eb799b381075f9ff7a7fd4c5cf2827ffac0c72abf123553d64b60a"
    results, comparison = fit_suite(
        data, direction_degrees=DIRECTION_DEGREES, p_values=P_VALUES,
        config=FIT_CONFIG, model_names=MODEL_NAMES, output_directory=run_directory,
    )
    print("Saved:", run_directory.resolve())
elif data is not None:
    print("Data loaded; set RUN_FITS=True to fit the configured models.")

## 9. Review all fits before selecting a model

The full CSV includes numerical and physical-unit parameters, actual `rho`,
`rho_fraction`, `rho_eff` for odd Matérn, fitted means, convergence flags and
search bounds reached. The parameter `rho` and `theta` refer to the configured
fixed reference unit. In a Gaussian mixed model, changing units rescales its
odd coefficient separately, so `rho` alone cannot be converted without `theta`.

The compact selection below excludes unconverged fits by default. If a model
has no converged fit, expand/restart its search rather than treating its absence
as evidence against the covariance family. A small relative likelihood gradient
alone does not establish global optimality or parameter identifiability.

In [ ]:
if not comparison.empty:
    from IPython.display import display
    columns = ["model", "p", "status", "nll", "aic", "bic", "k", "rho", "theta", "seconds", "bounds_reached"]
    display(comparison[[c for c in columns if c in comparison.columns]])
    print("Lowest AIC per model among converged fits:")
    display(selected_fits(comparison, criterion="aic"))
    figure = plot_model_comparison(comparison)
    if figure is not None and run_directory is not None:
        figure.savefig(run_directory / "aic_comparison.png", dpi=160, bbox_inches="tight")
else:
    print("No fits to summarize.")

## 10. Quick checks and an optional synthetic demonstration

These small checks catch normalization, zero-lag and matrix-orientation errors.
The optional demonstration exercises all six models on explicitly synthetic
data and writes no files. Its short optimizer budget is for a smoke test, not
an identifiability study or a replacement for the manuscript simulations.

In [ ]:
# p=1 Gaussian derivative identity along a non-axis direction.
check_geo = Geometry(np.array([[0.0, 0.0], [0.6, 0.8]]), 0.0)
check_a = 1.7
got = check_geo.expand(gaussian_radial(check_geo.unique_r, check_a, 1))*check_geo.angular(1)
expected = (2*0.6/check_a**2)*np.exp(-1/check_a**2)
np.testing.assert_allclose(got[0, 1], expected, rtol=1e-13)
np.testing.assert_allclose(got, -got.T, atol=1e-14)
np.testing.assert_allclose(matern_radial(np.array([0.0, 0.7]), 1.3, 0.5), [1.0, np.exp(-1.3*0.7)], rtol=1e-12)
assert matern_radial(np.array([0.0]), 1.0, 2.0, 3)[0] == 0.0
check_log = log_component_sup("matern", 1, 1, 1, nu11=0.5, nu22=0.5, nuc=1, p=1, odd=True)
np.testing.assert_allclose(np.exp(check_log), 4.0, rtol=2e-10)
print("Kernel normalization, odd parity and spectral-bound checks passed.")

if RUN_SYNTHETIC_DEMO:
    from IPython.display import display
    demo_data, demo_truth = synthetic_example(n_side=3, replicates=2)
    demo_config = FitConfig(seed=1729, n_candidates=8, n_starts=1, maxiter=20)
    demo_results, demo_comparison = fit_suite(
        demo_data, direction_degrees=20.0, p_values=(1,), config=demo_config,
        output_directory=None,
    )
    display(demo_comparison[["model", "status", "nll", "aic", "seconds"]])

## Formula and numerical references

- [NIST DLMF 10.22.51: Gaussian radial integral](https://dlmf.nist.gov/10.22.E51)
- [NIST DLMF 10.22.46: Matérn radial integral](https://dlmf.nist.gov/10.22.E46)
- [SciPy scaled Bessel K (`kve`)](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.kve.html)
- [SciPy Latin-hypercube sampling](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.qmc.LatinHypercube.html)
- [SciPy triangular solves](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.solve_triangular.html)

The proof corrections and rerun requirements discussed with this notebook apply
to the manuscript's numerical results. This file intentionally retains no old
figures, estimates or optimizer output.